# Lab 5 — Evaluator-Optimizer: translating an abstract until it passes the criteria (§19.1.5)

The teaching assistant at **Cogito Learn** must translate an English abstract into Thai for students,
with **the meaning complete** and **technical terms kept in English**

Evaluator-optimizer is a feedback loop between two models:
- ✍️ **Generator** writes a draft translation
- 🧐 **Evaluator** (separate) scores it against **explicitly written criteria**
- If the score is below the **threshold** → the **critique** is appended to the generator's context → loop again
  until it passes the criteria or the **retry budget** runs out

> Unlike Reflexion (EP05): here a separate evaluator checks against fixed criteria in a loop with a fixed number of rounds,
> while Reflexion keeps *lessons* in memory across many attempts

The data in this lab (the abstract) is fictional, but every model output comes from a real run

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.1.5 (p.370) — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **6 places marked `✏️ TODO`** (replace `___` or `...` with your code). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`19-1-lab5-evaluator.en.ipynb`** (the same code as in the video).

## 0. Setup

You need Python 3.10+ (or Colab) and an **OpenRouter API key** — setup instructions are in Lab 1

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

In [ ]:
# ---- from Lab 1: set up client + ask_llm ----
import os, json
from openai import OpenAI

def load_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv()
        return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"provider": {"require_parameters": True}, "reasoning": {"enabled": False}}

def ask_llm(prompt, temperature=0.7, schema=None):
    kw = {"response_format": {"type": "json_schema", "json_schema": {"name": "out", "strict": True, "schema": schema}}} if schema else {}
    resp = client.chat.completions.create(model=MODEL, extra_body=EXTRA, temperature=temperature,
                                          messages=[{"role": "user", "content": prompt}], **kw)
    text = resp.choices[0].message.content
    return json.loads(text) if schema else text

## 1. The task: abstract + glossary

In [ ]:
ABSTRACT = (
    "We study hallucination in small language models that answer students' questions about "
    "course material. We combine retrieval-augmented generation with lightweight fine-tuning on "
    "4,200 question-answer pairs from three university courses. On a new benchmark of 600 "
    "questions, our method reduces hallucination by 31% while keeping answer latency under "
    "two seconds. We release the benchmark and code for future work."
)

GLOSSARY = ["hallucination", "small language model", "retrieval-augmented generation",
            "fine-tuning", "benchmark", "latency"]

TASK = ("Translate this research abstract into Thai for Thai university students. "
        f"Reply with the translation only.\n\nAbstract:\n{ABSTRACT}")

## 2. Evaluator — scoring against explicitly written criteria

| Criterion | Score | Checked by |
|---|---|---|
| meaning — every claim, qualifier and number is present and correct | 0–4 | LLM |
| terms — glossary terms are in English, with no Thai translation | 0–4 | code |
| fluency — reads as natural Thai academic writing | 0–2 | LLM |

In [ ]:
def term_problems(draft):
    text = draft.lower()
    return [t for t in GLOSSARY
            if t not in text or f"({t}" in text or f"{t})" in text]

print(term_problems("reduce false-information generation (hallucination) with fine-tuning"))

In [ ]:
CRITERIA = """1. meaning (0-4): every claim, qualifier (e.g. small, lightweight) and number of the
   source is present and correct in the Thai draft.
2. fluency (0-2): reads like natural Thai academic writing.
English technical terms inside the Thai text are expected and correct; do not penalize them."""

In [ ]:
SCHEMA = {"type": "object", "additionalProperties": False,
          "required": ["meaning", "fluency", "critique"],
          "properties": {"meaning": {"type": "integer"}, "fluency": {"type": "integer"},
                         "critique": {"type": "string"}}}

In [ ]:
def evaluate(source, draft):
    review = ask_llm(
        "You are a strict editor of English-to-Thai translations.\n"
        f"Score the Thai draft against these criteria:\n{CRITERIA}\n\n"
        f"Source (English):\n{source}\n\nThai draft:\n{draft}\n\n"
        # ✏️ TODO 1: Call the evaluator with temperature 0 and force the answer to follow SCHEMA
        "In critique, list at most 3 concrete problems and how to fix each, one short line each "
        "(in English). If nothing is wrong, say so.",
        temperature=___, schema=___)
    # ✏️ TODO 2: Check the terms with code; if any are wrong, append the list of those terms to the critique
    bad = ___
    if bad:
        review["critique"] += ___
    # ✏️ TODO 3: Compute the terms score (4 minus 2 per bad term, never below 0), then add the three criteria into a score out of 10
    review["bad_terms"] = bad
    review["terms"] = ___
    review["score"] = ___
    return review

## 3. Evaluator-optimizer loop

`generate → evaluate → (passed threshold? stop) → append critique to context → generate again` for at most `budget` rounds

In [ ]:
def evaluator_optimizer(source, threshold=9, budget=3):
    context = TASK
    for rnd in range(1, budget + 1):
        # ✏️ TODO 4: Each round, have the generator write a draft from the context, then send the draft to the evaluator
        draft = ___
        review = ___
        print(f"\n--- round {rnd}: score {review['score']}/10 "
              f"(meaning {review['meaning']}, terms {review['terms']}, fluency {review['fluency']})")
        print(draft)
        print("❌ bad terms:", review["bad_terms"] or "none")
        print("🧐 critique:", review["critique"])
        # ✏️ TODO 5: If the score reaches the threshold, stop and return this draft
        if ___:
            print(f"✅ passed in round {rnd}")
            return draft
        # ✏️ TODO 6: Append the previous draft + the evaluator's critique to the generator's context
        context += ___
    print(f"⛔ retry budget of {budget} rounds used up, returning the last draft")
    return draft

In [ ]:
final = evaluator_optimizer(ABSTRACT)

## Summary

| Component (§19.1.5) | In the code |
|---|---|
| Generator | `ask_llm(context, temperature=0.7)` |
| Separate evaluator + explicit criteria | `evaluate()` + `CRITERIA` + `SCHEMA`, temperature 0 + `term_problems()` (terms checked by code) |
| Threshold | `threshold=9` |
| Critique appended to context | `context += ...` |
| Retry budget | `budget=3` |

**Evaluator-optimizer vs Reflexion (EP05):** here a separate evaluator checks against explicit criteria in a loop with a fixed number of rounds,
while Reflexion keeps lessons in memory across many attempts

**Try next:** set `threshold=10, budget=1` and see how the guard works,
or add a new term to `GLOSSARY` (e.g. "question-answer pairs") and see how the critique changes